# LightGBM Regressor: Raw Price Prediction (Ablation Study)
This experiment trains a LightGBM model to predict the raw `price` directly instead of `log_return`. 

**Goal:** See how the model performs without the stationarity provided by log returns.

In [ ]:
import os
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
from pathlib import Path

# Handle paths based on execution directory
if Path.cwd().name == 'notebooks':
    base_dir = Path('..')
else:
    base_dir = Path('.')

data_dir = base_dir / 'data' / 'processed'
assets_dir = base_dir / 'assets'
assets_dir.mkdir(parents=True, exist_ok=True)

train_path = data_dir / 'train.csv'
test_path = data_dir / 'test.csv'

print(f"Reading data from: {data_dir}")

## 1. Feature Engineering: Price Lags
Since the existing `lag_n` features are built on `log_return`, we must recreate them based on the `price` column for this study.

In [ ]:
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

train_df['date'] = pd.to_datetime(train_df['date'])
test_df['date'] = pd.to_datetime(test_df['date'])

# Ensure chronological order
train_df = train_df.sort_values(['RegionID', 'date']).reset_index(drop=True)
test_df = test_df.sort_values(['RegionID', 'date']).reset_index(drop=True)

LAG_PERIODS = [1, 2, 3, 6, 12]

def add_price_lags(df):
    df = df.copy()
    for lag in LAG_PERIODS:
        df[f'price_lag_{lag}'] = df.groupby('RegionID')['price'].shift(lag)
    return df

train_price = add_price_lags(train_df).dropna().reset_index(drop=True)
test_price = add_price_lags(test_df).ffill().reset_index(drop=True)

features = [f'price_lag_{l}' for l in LAG_PERIODS] + ['city_enc']
target = 'price'

# Time-based Validation Split (80/20 split as requested)
unique_dates = train_price['date'].sort_values().unique()
split_idx = int(len(unique_dates) * 0.80)
split_date = unique_dates[split_idx]

train_subset = train_price[train_price['date'] < split_date]
val_subset = train_price[train_price['date'] >= split_date]

X_train_sub = train_subset[features]
y_train_sub = train_subset[target]
X_val = val_subset[features]
y_val = val_subset[target]
X_test = test_price[features]
y_test = test_price[target]

print(f"Training rows: {len(X_train_sub)}")
print(f"Validation rows: {len(X_val)}")

## 2. Training
We use identical hyper-parameters to the log-return model for a fair comparison.

In [ ]:
scaler = StandardScaler()
X_train_sub_scaled = scaler.fit_transform(X_train_sub)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train_sub_scaled, y_train_sub,
    eval_set=[(X_val_scaled, y_val)],
    eval_metric='rmse',
    callbacks=[lgb.early_stopping(stopping_rounds=50)]
)

## 3. Evaluation & Visualization

In [ ]:
y_pred = model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"\n[Raw Price Model] MAE: ${mae:.2f} | RMSE: ${rmse:.2f} | R2: {r2:.6f}")

# Create comparison dataframe for date-based visualization
results_df = pd.DataFrame({
    'date': test_price['date'],
    'actual': y_test,
    'predicted': y_pred
})

# Aggregate by date to see the general trend across all regions
daily_results = results_df.groupby('date').mean().reset_index()

plt.figure(figsize=(14, 7))
plt.plot(daily_results['date'], daily_results['actual'], label='Average Actual Price', color='#2ecc71', linewidth=2, marker='o', markersize=4)
plt.plot(daily_results['date'], daily_results['predicted'], label='Average Predicted Price', color='#e74c3c', linestyle='--', linewidth=2)
plt.title('House Price Trends: Actual vs Predicted (Raw Price Model)')
plt.xlabel('Date')
plt.ylabel('Average Price ($)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(assets_dir / 'raw_price_predictions_date.png', dpi=300)
plt.show()